# Building GPT

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
words = open("input.txt").read()

# now, we build the vocabulary.
# vocabulary is nothing but how many unique characters are there in the input text. we will use this to encode the text into integers.

vocab = sorted(list(set(words)))
vocab_size = len(vocab)
print(''.join(vocab))

In [ ]:
# we're creating a mapping from char to int, and vice-versa.

stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a strin

## Making our dataset now.

### We're going to be splitting this into train and val sets.
### train - 90%, val - 10%.

In [ ]:
data = torch.tensor(encode(words), dtype=torch.long)
print(data.shape, data.dtype)

# now that we have the data in the form of a tensor
# we want to split this into train and val sets.

n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
# get_batch basically just provides a single batch of inputs to the network.
# it creates the inputs, and also the targets against which the inputs will be evaluated (loss).

torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embed = 32
n_head = 4
head_size = 32 # 32 for a single head of self-attention, 8 for multi-head (4) attention.

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

## Introducing Attention
### how are we going to introduce attention?
### so, first, we want each token to be able to talk to all of it's previous tokens
### so, we use masking a lower triangular matrix, and then softmax it
### but, this gives us a uniform representation of all tokens (i.e. all tokens from t_0 to t_m have the same affinity)


In [ ]:
# so, attention is supposed to be implemented on input tokens
# X (token_embeds + position_embeds) is the input to the attention layer, 
# and the output of the attention layer is then passed through a linear layer to get the logits.

token_embedding_table = nn.Embedding(vocab_size, n_embed) # (65, 32)
position_embedding_table = nn.Embedding(block_size, n_embed) # (32, 32
token_embeds = token_embedding_table(torch.tensor([[0, 1, 2, 3]])) # (1, 4, 32)
position_embeds = position_embedding_table(torch.arange(4)) # (4, 32)

X = token_embeds + position_embeds # (1, 4, 32)

# implementing a rudimentary version of attention now
mask = torch.zeros((block_size, block_size)) # (32, 32)
mask = mask.masked_fill(torch.tril(torch.ones(block_size, block_size)) == 0, float('-inf'))

wei = F.softmax(mask, dim=-1) # (32, 32)

In [ ]:
wei.shape

In [65]:
class SingleHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_q = nn.Linear(n_embed, head_size, bias=False)
        self.W_k = nn.Linear(n_embed, head_size, bias=False)
        self.W_v = nn.Linear(n_embed, head_size, bias=False)

    def forward(self, input):
        B, T, C = input.shape
        query = self.W_q(input) # (B, T, head_size)
        key = self.W_k(input) # (B, T, head_size)
        value = self.W_v(input) # (B, T, head_size)

        scores = query @ torch.transpose(key, 1, 2) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)

        # now, we want to apply the causal mask to the scores matrix.
        scores = scores.masked_fill(torch.tril(torch.ones(T, T, device=device)) == 0, float('-inf'))
        scores = F.softmax(scores, dim=-1)
        out = scores @ value # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)

        return out


In [ ]:
# now, we're going to build up an embedding lookup table.
# this is basically a giant table that contains the embedding vectors for each character in the vocab.
# so, this is going to be of shape (65, 32).
# 65 because we have 65 unique chars in the vocab, and 32 as the embedding dimension because we want each char to be represented as a 32-dimensional vector.

# we also want to add positional embeddings, which will help the model understand the order of the characters in the input sequence.
# because, the order matters too. "cat in a hat" is different from "hat in a cat".
# and how are these gotten and represented?
# similar to the token embeddings, we can have a posi embedding block of tunable parameters, of shape (block_size, n_embed).
# so, logits (without batches) are going to be (T, n_embed), and the posi embeddings are going to be (T, n_embed) as well, so we can just add them together.


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # (65, 32)
        self.position_embedding_table = nn.Embedding(block_size, n_embed) # (8, 32)
        self.lm_head = nn.Linear(n_embed, vocab_size) # (32, 65)
        self.attn = SingleHeadAttention()
        self.W_q = nn.Linear(n_embed, head_size, bias=False)
        self.W_k = nn.Linear(n_embed, head_size, bias=False)
        self.W_v = nn.Linear(n_embed, head_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeds = self.token_embedding_table(idx) # (B, T, C)
        position_embeds = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = token_embeds + position_embeds # (B, T, C)

        # now, we want to include the Q, K, V layers here.

        out = self.attn(x)
        logits = self.lm_head(out) # (B, T, head_size)

        # now, we want to do a forward pass, which is just calculating the logits, and the loss
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [67]:
m = BigramLanguageModel()
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# the initial loss, before any backprop even happens is 4.6695.
# now, let's automate the forward & backward passes.

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    print(f"Step {steps}, Loss: {loss.item()}")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Step 0, Loss: 4.302004814147949
Step 1, Loss: 4.242160320281982
Step 2, Loss: 4.202601432800293
Step 3, Loss: 4.089661121368408
Step 4, Loss: 4.30841064453125
Step 5, Loss: 4.191402912139893
Step 6, Loss: 4.219326972961426
Step 7, Loss: 4.034656047821045
Step 8, Loss: 4.176311492919922
Step 9, Loss: 4.127935886383057
Step 10, Loss: 4.183750629425049
Step 11, Loss: 4.157492637634277
Step 12, Loss: 3.962484121322632
Step 13, Loss: 4.177571773529053
Step 14, Loss: 4.086996078491211
Step 15, Loss: 4.044526100158691
Step 16, Loss: 3.886946439743042
Step 17, Loss: 4.027297019958496
Step 18, Loss: 4.0392374992370605
Step 19, Loss: 3.9804325103759766
Step 20, Loss: 3.939183473587036
Step 21, Loss: 4.015923976898193
Step 22, Loss: 4.005350589752197
Step 23, Loss: 3.994610071182251
Step 24, Loss: 4.090902328491211
Step 25, Loss: 3.9216508865356445
Step 26, Loss: 3.9566426277160645
Step 27, Loss: 4.016730785369873
Step 28, Loss: 4.017977237701416
Step 29, Loss: 4.104063034057617
Step 30, Loss: 3.

In [ ]:
# now, how're we going to generate output from the trained model?
# we have to get the output's, i.e. the softmax probabilities, and keep picking the most probable out of them at every turn
# we have to rememeber that we're not training here, just generating. so, no backprop.

context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
